# Trimorious 2.0 | M2.3c — Private upload, integrity verification and public release

**Notebook 07 of 07 · CPU runtime**

**Before running:** inspect notebook 06's package. Create a fine-grained Hugging Face write token for your account, add it as `HF_TOKEN` in Colab Secrets, enable notebook access. Keep the notebook private and never print the token. Set `DATA_LICENSE_REVIEWED=True`, `HUMAN_EVAL_APPROVED=True` (only after actual review), and `PUBLISH_APPROVAL='PUBLISH TRIMORIOUS 2.0'`. Leave `MAKE_PUBLIC=False` for the first run. The code creates a new private repository `roime81/trimorious-research-4B-v2`, uploads only allowlisted release files and downloads them to verify hashes. It will not overwrite an unexpected existing repository. If the creation/upload was interrupted, inspect the repository and receipt; do not delete or overwrite to bypass checks.

After the verified private repository is inspected, rerun this same notebook with `MAKE_PUBLIC=True` and `PUBLIC_APPROVAL='MAKE TRIMORIOUS 2.0 PUBLIC'`. The original v1 and source datasets remain unchanged. No private benchmark, dataset, teacher logs or tokens are uploaded. Do not enable public visibility if any gate fails. The playground can be created after the model is published.

All stages use the same `MyDrive/trimorious_PRIVATE_m1_4_1/release_v2/` workspace. Save this notebook in Drive. Edit only the configuration and select Runtime → Run all. Never run another GPU notebook concurrently. Each notebook is self-contained and contains no credentials. GPU execution is not locally tested.

In [ ]:
# @title Configuration — PUBLISH
STAGE = 'PUBLISH'
USE_DRIVE = True # @param {type:'boolean'}
INSTALL_GPU_DEPS = True # @param {type:'boolean'}
PROJECT_ROOT = '/content/drive/MyDrive/trimorious_PRIVATE_m1_4_1'
SOURCE_NAME = 'trimorious_m1_4_1_PRIVATE_GPU_partial_20261127T061605946898Z.zip'
D07_FILENAME = 'trimorious_m1_3_2_PRIVATE_full_1c3d70c27cc3c9f8.zip'
# Save this benchmark ZIP in PROJECT_ROOT (or upload to /content once). Required for EVALUATE.
FRESH_TEST_PATH = '' # @param {type:'string'}
# Fresh test JSONL format documented in section 5. Must be externally authored, sealed before evaluation; preferably before training.
FRESH_TEST_SHA256 = '' # @param {type:'string'}
FRESH_INDEPENDENCE_ATTESTED = False # @param {type:'boolean'}
# Confirm external authorship, expert key review, no access to this adapter's evaluation outputs, no tuning on the test.
DATA_LICENSE_REVIEWED = False # @param {type:'boolean'}
DATA_REVIEW_APPROVED = False # @param {type:'boolean'}
# By approving, you confirm rights to train and distribute the resulting adapter, and actual review of CSV rows.
HUMAN_EVAL_APPROVED = False # @param {type:'boolean'}
# After actual rubric review; see evaluation report. Not an automatic pass.
PUBLISH_APPROVAL = '' # @param {type:'string'}
MAKE_PUBLIC = False # @param {type:'boolean'}
PUBLIC_APPROVAL = '' # @param {type:'string'}
MAX_TRAIN_PER_TEMPLATE = 12
MAX_VAL_PER_TEMPLATE = 6
MAX_LENGTH = 512
GRAD_ACCUM = 16
EPOCHS = 1
SEED = 417031
# Provisional minimum release policy: approve before training; cannot change after dataset freeze.
MIN_D07_GAIN = 1.0       # percentage points, measurable diagnostic gain
MAX_FAMILY_DROP = 5.0   # percentage points
MIN_FRESH_GAIN = 0.0
MAX_PROTOCOL_DROP = 2.0
BASE = 'Qwen/Qwen3-4B-Instruct-2507'
REV = 'cdbee75f17c01a7cc42f958dc650907174af0554'
TARGET = 'roime81/trimorious-research-4B-v2'
EXPECTED_CORPUS = '9fe5e9dbce17096c4d1ddd28a82e45111a37cbba66c574ec63473f6f70dd283a'

In [ ]:
# @title Mount Drive and prepare local worker scripts
from pathlib import Path
import json, os, sys, subprocess, hashlib, csv, sqlite3, shutil, traceback
from collections import Counter
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
ROOT = Path(PROJECT_ROOT)
assert ROOT.is_dir(), f'Project folder missing: {ROOT}'
WORK = ROOT/'release_v2'; WORK.mkdir(exist_ok=True)
SCRIPT_DIR = Path('/tmp/trimorious_v2_sequence'); SCRIPT_DIR.mkdir(exist_ok=True)
print('Persistent workspace:', WORK)
print('Local script directory:', SCRIPT_DIR)
CONFIG_NAMES=['USE_DRIVE', 'INSTALL_GPU_DEPS', 'PROJECT_ROOT', 'SOURCE_NAME', 'D07_FILENAME', 'FRESH_TEST_PATH', 'FRESH_TEST_SHA256', 'FRESH_INDEPENDENCE_ATTESTED', 'DATA_LICENSE_REVIEWED', 'DATA_REVIEW_APPROVED', 'HUMAN_EVAL_APPROVED', 'PUBLISH_APPROVAL', 'MAKE_PUBLIC', 'PUBLIC_APPROVAL', 'MAX_TRAIN_PER_TEMPLATE', 'MAX_VAL_PER_TEMPLATE', 'MAX_LENGTH', 'GRAD_ACCUM', 'EPOCHS', 'SEED', 'MIN_D07_GAIN', 'MAX_FAMILY_DROP', 'MIN_FRESH_GAIN', 'MAX_PROTOCOL_DROP', 'BASE', 'REV', 'TARGET', 'EXPECTED_CORPUS']
CONFIG={k:globals()[k] for k in CONFIG_NAMES}
assert TARGET=='roime81/trimorious-research-4B-v2' and BASE=='Qwen/Qwen3-4B-Instruct-2507' and REV=='cdbee75f17c01a7cc42f958dc650907174af0554'

## Embedded implementation
The following cells write the implementation to the local runtime. Run them unchanged. No source code download required.

In [ ]:
%%writefile /tmp/trimorious_v2_sequence/common.py
# @title 1. Mount Drive and common helpers (CPU)
import os, sys, json, hashlib, math, re, random, csv, zipfile, sqlite3, statistics, time, gc, shutil, platform
from pathlib import Path
from collections import Counter, defaultdict
from fractions import Fraction
from importlib import metadata
assert STAGE in {'RECOVER','APPROVE','SMOKE','TRAIN','EVALUATE','PACKAGE','PUBLISH'}
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
ROOT=Path(PROJECT_ROOT); assert ROOT.is_dir(), f'Missing project folder: {ROOT}'
WORK=ROOT/'release_v2'; WORK.mkdir(exist_ok=True)
def canon(x): return json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(',',':'),allow_nan=False)
def sha(b): return hashlib.sha256(b if isinstance(b,bytes) else canon(b).encode()).hexdigest()
def put(p,obj):
    p=Path(p); p.parent.mkdir(parents=True,exist_ok=True); tmp=p.with_suffix(p.suffix+'.tmp'); tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,allow_nan=False)); tmp.replace(p)
def get(p): return json.loads(Path(p).read_text())
def lines(b): return [json.loads(x) for x in b.splitlines() if x.strip()]
def readl(p): return lines(Path(p).read_text())
def writel(p,rows):
    p=Path(p); tmp=p.with_suffix(p.suffix+'.tmp'); tmp.write_text(''.join(canon(r)+'\n' for r in rows)); tmp.replace(p)
def writecsv(p,rows,fields):
    with Path(p).open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=fields); w.writeheader(); w.writerows(rows)
def archive_check(z):
    ns=z.namelist(); assert len(ns)==len(set(ns)) and z.testzip() is None
    m=json.loads(z.read('archive_manifest.json')); assert set(m)==set(ns)-{'archive_manifest.json'}
    assert all(sha(z.read(k))==v for k,v in m.items()), 'Archive hash mismatch'
    return m
def state(stage,**kw):
    p=WORK/'project_state.json'; s=get(p) if p.exists() else {}; s[stage]=kw; put(p,s)
def norm(s): return re.sub(r'\s+',' ',s.casefold()).strip()
def mask(s): return re.sub(r'(?<![a-z])[-+]?\d+(?:\.\d+)?(?:e[-+]?\d+)?','NUM',norm(s))
def close(a,b): return math.isclose(float(a),float(b),rel_tol=1e-8,abs_tol=1e-10)
POLICY={'min_d07_gain_pp':MIN_D07_GAIN,'max_family_drop_pp':MAX_FAMILY_DROP,'min_fresh_gain_pp':MIN_FRESH_GAIN,'max_protocol_drop_pp':MAX_PROTOCOL_DROP}
print('Stage:',STAGE,'| Persistent work directory:',WORK)

In [ ]:
%%writefile /tmp/trimorious_v2_sequence/approve.py
def load_approved():
    approval=get(WORK/'dataset_approval.json'); assert approval['training_approved'] is True
    for name,digest in approval['files'].items(): assert sha((WORK/name).read_bytes())==digest, f'Changed approved file: {name}'
    assert approval['policy']==POLICY and approval['base']==BASE and approval['revision']==REV
    return approval

if STAGE=='APPROVE':
    assert DATA_LICENSE_REVIEWED and DATA_REVIEW_APPROVED, 'Review rights and examples first; then set both approval flags.'
    report=get(WORK/'recovery.json'); candidates=readl(WORK/'review_candidates.jsonl'); assert sha(candidates)==report['review_set_sha256']
    with (WORK/'review_queue.csv').open(newline='',encoding='utf-8-sig') as f: reviews=list(csv.DictReader(f))
    by_id={r['id']:r for r in reviews}; assert len(reviews)==len(by_id)==len(candidates) and set(by_id)=={r['id'] for r in candidates}
    approved={'train':[],'validation':[]}; dispositions=[]
    for r in candidates:
        q=by_id[r['id']]; assert q['prompt']==r['prompt'] and q['response']==r['reference_response'] and q['split']==r['split'] and q['template']==r['template_id']
        d=q['decision'].strip().lower(); assert d in {'','approve','reject'}
        if d: assert q['reviewer'].strip() and q['notes'].strip(), 'Each explicit decision requires reviewer and rationale.'
        dispositions.append({'id':r['id'],'decision':d or 'unreviewed','reviewer':q['reviewer'],'notes':q['notes']})
        if d=='approve':
            assert close(recompute(r),r['numeric_answer']) and not r['d07_screen']['flagged']
            # Use the source response as written, not unapproved teacher output.
            split='train' if r['split']=='development_candidate' else 'validation'
            approved[split].append({'id':r['id'],'template':r['template_id'],'family':r['family'],'messages':[{'role':'user','content':r['prompt']},{'role':'assistant','content':r['reference_response']}]})
    assert len(approved['train'])>=32 and len(approved['validation'])>=8, 'Review at least 32 development and 8 validation examples; these are smoke-level minima, not a quality guarantee.'
    assert {r['template'] for r in approved['train']}.isdisjoint({r['template'] for r in approved['validation']})
    assert not ({mask(r['messages'][0]['content']) for r in approved['train']} & {mask(r['messages'][0]['content']) for r in approved['validation']})
    if FRESH_TEST_SHA256: assert re.fullmatch('[a-f0-9]{64}',FRESH_TEST_SHA256)
    data_names=['train.jsonl','validation.jsonl','review_dispositions.jsonl']
    contents=[approved['train'],approved['validation'],dispositions]
    files={name:sha(''.join(canon(r)+'\n' for r in rows).encode()) for name,rows in zip(data_names,contents)}
    approval={'training_approved':True,'license_review_attested':True,'corpus_sha256':EXPECTED_CORPUS,'review_set_sha256':report['review_set_sha256'],'files':files,'counts':{k:len(v) for k,v in approved.items()},'policy':POLICY,'fresh_test_sha256':FRESH_TEST_SHA256,'base':BASE,'revision':REV,'scope':'human-reviewed, recomputed synthetic math/science/statistics; no teacher or tool trace SFT'}
    if (WORK/'dataset_approval.json').exists(): assert get(WORK/'dataset_approval.json')==approval, 'Approval already frozen. New revision requires separate workspace.'
    for name,rows in zip(data_names,contents): writel(WORK/name,rows)
    put(WORK/'dataset_approval.json',approval); state('APPROVE',complete=True,counts=approval['counts']); print(json.dumps(approval,indent=2)); print('NEXT: SMOKE')

In [ ]:
%%writefile /tmp/trimorious_v2_sequence/eval_setup.py
def active_run():
    approval=load_approved(); p=WORK/'sft_output'/get(WORK/'active_run.json')['run']; done=get(p/'training_complete.json')
    assert done['config']['approval_sha256']==sha(approval)
    final=p/'final_adapter'
    assert all(sha((final/k).read_bytes())==v for k,v in done['adapter_hashes'].items()), 'Adapter changed'
    return approval,p,done,final

if STAGE=='EVALUATE':
    approval,run_dir,training_done,final=active_run()
    assert not (approval['fresh_test_sha256'] or (run_dir/'fresh_test_seal.json').exists()) or FRESH_TEST_PATH, 'A sealed fresh test is configured. Provide its path to avoid repeating D07 evaluation.'
    d07=ROOT/D07_FILENAME
    if not d07.exists(): d07=Path('/content')/D07_FILENAME
    assert d07.is_file(), 'Upload original M1.3.2 full archive.'
    with zipfile.ZipFile(d07) as z:
        archive_check(z); pre='PRIVATE_DO_NOT_TRAIN/'; m=json.loads(z.read(pre+'manifest.json'))
        assert m['training_use'] is False and all(sha(z.read(pre+k))==v for k,v in m['file_hashes'].items())
        questions=lines(z.read(pre+'D07_questions.jsonl')); keys=lines(z.read(pre+'D07_gold_key.jsonl')); keymap={r['id']:r for r in keys}
        assert len(questions)==len(keys)==len(keymap)==1000
        assert all(all(keymap[q['id']].get(k,v)==v for k,v in q.items()) for q in questions)
        benchmark=[{**q,**keymap[q['id']]} for q in questions]
        assert sha(benchmark)==m['benchmark_sha256']=='47694f1b2f4dbd44e8f5b6e58962ad07af5c2a844e375249e74d785de0b51283'
        assert sha(z.read(pre+'D07_questions.jsonl'))=='e724980d46b644f47952d3ab87789e94a32cb0b79835c4721ad3c7616c311c7e'
    fresh=[]
    if FRESH_TEST_PATH:
        fresh_bytes=Path(FRESH_TEST_PATH).read_bytes(); fresh_digest=sha(fresh_bytes)
        assert FRESH_INDEPENDENCE_ATTESTED, 'Confirm independent authorship and answer-key review before evaluation.'
        expected=approval['fresh_test_sha256'] or FRESH_TEST_SHA256
        assert expected and fresh_digest==expected, 'Fresh test hash missing or mismatch.'
        assert not approval['fresh_test_sha256'] or not FRESH_TEST_SHA256 or FRESH_TEST_SHA256==expected
        seal={'sha256':fresh_digest,'adapter_hashes':training_done['adapter_hashes'],'independence_attested':True,'sealed_before_training':bool(approval['fresh_test_sha256'])}
        seal_path=run_dir/'fresh_test_seal.json'
        if seal_path.exists(): assert get(seal_path)==seal, 'Fresh confirmation already sealed for this frozen adapter.'
        fresh=lines(fresh_bytes); assert len(fresh)>=100 and len({r['template'] for r in fresh})>=10
        assert len({r['id'] for r in fresh})==len(fresh) and len({norm(r['question']) for r in fresh})==len(fresh)
        for r in fresh:
            assert r['kind'] in {'numeric','choice'} and all(isinstance(r[k],str) and r[k].strip() for k in ['id','question','template','family'])
            if r['kind']=='numeric': assert isinstance(r['gold'],(int,float)) and not isinstance(r['gold'],bool) and math.isfinite(r['gold']) and 0<=r['atol']<=.01 and 0<=r['rtol']<=.001
            else: assert r['gold'] in 'ABCD' and len(r['gold'])==1
        known=[r['messages'][0]['content'] for name in ['train.jsonl','validation.jsonl'] for r in readl(WORK/name)]+[r['question'] for r in benchmark]
        assert not ({mask(q) for q in known}&{mask(r['question']) for r in fresh}), 'Fresh masked question overlap'
        assert not ({r['id'] for r in fresh}&{r['id'] for r in benchmark})
        # Check lexical similarity without using gold. This is not semantic independence proof.
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.metrics.pairwise import cosine_similarity
        v=TfidfVectorizer(analyzer='char',ngram_range=(3,5)); x=v.fit_transform(known+[r['question'] for r in fresh]); sim=cosine_similarity(x[len(known):],x[:len(known)]).max()
        assert sim<.78,f'Fresh test near-overlap: {sim}'
        put(seal_path,seal)
    evaluation_config={'base':BASE,'revision':REV,'adapter_hashes':training_done['adapter_hashes'],'versions':versions,'benchmark_hash':sha(benchmark),'fresh_hash':fresh_digest if fresh else '', 'max_new_tokens':384,'seed':132,'attention':'eager','precision':'nf4_fp16','scorer':'v0.2'}
    eval_dir=run_dir/'evaluation'/sha(evaluation_config)[:20]; eval_dir.mkdir(parents=True,exist_ok=True); put(eval_dir/'config.json',evaluation_config)
    print('Evaluation rows:',len(benchmark),'+',len(fresh),'fresh; checkpoints:',eval_dir)

In [ ]:
%%writefile /tmp/trimorious_v2_sequence/package.py
if STAGE in {'PACKAGE','PUBLISH'}:
    approval,run_dir,training_done,final=active_run()
    active=get(WORK/'active_evaluation.json'); assert active['run']==run_dir.name
    eval_dir=run_dir/'evaluation'/active['evaluation']; report=get(eval_dir/'evaluation_report.json')
    assert report['evaluation_config']['adapter_hashes']==training_done['adapter_hashes']
    assert report['objective_gate_passed'] and report['fresh'], 'Objective/independent confirmation gate failed. No publication.'
    seal=get(run_dir/'fresh_test_seal.json'); assert seal['sha256']==report['evaluation_config']['fresh_hash'] and seal['adapter_hashes']==training_done['adapter_hashes'] and seal['independence_attested']
    if approval['fresh_test_sha256']: assert seal['sha256']==approval['fresh_test_sha256']
    assert HUMAN_EVAL_APPROVED, 'Actual human scientific/rubric review and independent confirmation attestation required.'
    with (eval_dir/'rubric_review_private.csv').open(newline='',encoding='utf-8-sig') as f: rows=list(csv.DictReader(f))
    mapping=get(eval_dir/'rubric_mapping_private.json'); assert len(rows)==len(mapping)==120 and {r['id'] for r in rows}==set(mapping)
    # Ensure the reviewed text is the exact saved inference, not modified answers.
    with sqlite3.connect(eval_dir/'responses.sqlite3') as conn:
        reviewed_counts=Counter()
        for r in rows:
            assert r['reviewer'].strip() and r['notes'].strip()
            for col,label in zip(['a','b'],mapping[r['id']]):
                data=json.loads(conn.execute('SELECT payload FROM responses WHERE model=? AND item=?',(label,r['id'])).fetchone()[0]); assert r['answer_'+col]==data['response']
                ans=r[col+'_acceptable'].strip().lower(); assert ans in {'yes','no'}
                reviewed_counts[label]+=ans=='yes'
    rubric_pct={k:100*reviewed_counts[k]/120 for k in ['qwen3_base','trimorious_v2']}
    assert rubric_pct['trimorious_v2']-rubric_pct['qwen3_base']>=-POLICY['max_family_drop_pp'], 'Human rubric regression'
    assert DATA_LICENSE_REVIEWED, 'Reconfirm licensing and redistribution rights before packaging/publication.'
    package=run_dir/'release_package'; package.mkdir(exist_ok=True)
    if STAGE=='PACKAGE':
        allowed={'adapter_model.safetensors','adapter_config.json','tokenizer.json','tokenizer_config.json','special_tokens_map.json','added_tokens.json','vocab.json','merges.txt','chat_template.jinja'}
        for f in final.iterdir():
            if f.is_file() and f.name in allowed: shutil.copy2(f,package/f.name)
        assert (package/'adapter_model.safetensors').is_file() and (package/'adapter_config.json').is_file()
        # Fetch base license at the exact revision; fail closed if unavailable.
        import urllib.request
        with urllib.request.urlopen(f'https://huggingface.co/{BASE}/resolve/{REV}/LICENSE',timeout=60) as f: lic=f.read()
        assert b'Apache License' in lic and b'Version 2.0' in lic
        (package/'LICENSE').write_bytes(lic)
        public_report={'d07':report['d07'],'families':report['families'],'fresh':report['fresh'],'protocol_pct':report['protocol_pct'],'rubric_acceptance_pct':rubric_pct,'runtime_metrics':report['runtime_metrics'],'caveats':['D07 is a known synthetic diagnostic benchmark; not a blinded final test.','Fresh benchmark independence and review attested by user; sealed before training: '+str(seal['sealed_before_training']),'No native tool-call SFT, narrative teacher SFT or external literature corpus.','No claims of comprehensive scientific expertise.']}
        put(package/'benchmark_summary.json',public_report)
        put(package/'training_config.json',training_done['config']); (package/'requirements.txt').write_text('\n'.join(k+'=='+v for k,v in training_done['config']['versions'].items() if k!='torch')+'\n')
        card=f"""---
license: apache-2.0
base_model: {BASE}
base_model_relation: adapter
pipeline_tag: text-generation
library_name: peft
tags:
- scientific-research
- qlora
- experimental
---
# Trimorious 2.0 — experimental scientific LoRA adapter

Base: `{BASE}` at `{REV}`. This is a PEFT adapter, not standalone model weights. Load the exact base revision. Training uses 4-bit NF4 QLoRA, FP16 compute, assistant-only loss and a template-capped human-reviewed synthetic corpus ({approval['counts']['train']} train / {approval['counts']['validation']} validation). No teacher proposals or tool traces were used for SFT. No DPO or GRPO. The original Trimorious v1 is unchanged.

## Evaluation
D07 objective accuracy: base {report['d07']['base_pct']:.2f}% vs adapter {report['d07']['v2_pct']:.2f}% ({report['d07']['n']} tasks). Fresh confirmation: base {report['fresh']['base_pct']:.2f}% vs adapter {report['fresh']['v2_pct']:.2f}% ({report['fresh']['n']} tasks). Human rubric acceptance: base {rubric_pct['qwen3_base']:.2f}% vs adapter {rubric_pct['trimorious_v2']:.2f}%. See `benchmark_summary.json` for family scores, uncertainty, latency and limitations. No benchmark questions or answer keys are public.

## Usage
See `inference.py`. Requires a CUDA-compatible GPU and matching dependencies; use compatible PyTorch. Base model attribution: Qwen Team, Apache-2.0. The user has confirmed synthetic dataset rights and redistribution approval. Training data and private review artifacts are not distributed; full independent reproduction therefore requires authorized access.

## Limitations
Experimental, narrowly fine-tuned model; may hallucinate or regress on unseen problems. The known D07 benchmark is synthetic and not independent. Fresh confirmation and human review are limited in scope. Do not rely on outputs as sole scientific, clinical, engineering or safety-critical evidence. Verify calculations, sources and assumptions independently. Native tool calling is not newly trained or certified.
"""
        (package/'README.md').write_text(card); (package/'CHANGELOG.md').write_text('# Trimorious 2.0\n\n- New Qwen3-based experimental QLoRA adapter; original model retained.\n- Private diagnostic and independent confirmation evaluation completed.\n')
        inference=f"""import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
BASE={BASE!r}
REV={REV!r}
ADAPTER={TARGET!r}
tok=AutoTokenizer.from_pretrained(BASE,revision=REV)
q=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
base=AutoModelForCausalLM.from_pretrained(BASE,revision=REV,quantization_config=q,device_map={{'':0}},torch_dtype=torch.float16)
model=PeftModel.from_pretrained(base,ADAPTER).eval()
prompt=tok.apply_chat_template([{{'role':'user','content':'Explain the difference between correlation and causation.'}}],tokenize=False,add_generation_prompt=True)
inputs=tok(prompt,return_tensors='pt',add_special_tokens=False).to(model.get_input_embeddings().weight.device)
with torch.inference_mode(): out=model.generate(**inputs,max_new_tokens=256,do_sample=False,pad_token_id=tok.eos_token_id)
print(tok.decode(out[0,inputs['input_ids'].shape[1]:],skip_special_tokens=True))
"""
        (package/'inference.py').write_text(inference)
        public_names=allowed|{'LICENSE','benchmark_summary.json','training_config.json','requirements.txt','README.md','CHANGELOG.md','inference.py'}
        assert all(p.is_file() and p.name in public_names|{'artifact_manifest.json'} for p in package.iterdir())
        manifest={p.name:sha(p.read_bytes()) for p in package.iterdir() if p.name!='artifact_manifest.json'}; put(package/'artifact_manifest.json',manifest)
        put(run_dir/'package_approval.json',{'manifest_sha256':sha((package/'artifact_manifest.json').read_bytes()),'evaluation_sha256':sha(report),'rubric_review_sha256':sha((eval_dir/'rubric_review_private.csv').read_bytes()),'target':TARGET}); state('PACKAGE',complete=True,package=str(package)); print('RELEASE PACKAGE READY:',package,'| review files before PUBLISH')

In [ ]:
%%writefile /tmp/trimorious_v2_sequence/publish.py
if STAGE=='PUBLISH':
    assert PUBLISH_APPROVAL=='PUBLISH TRIMORIOUS 2.0', 'Explicit upload approval required.'
    assert TARGET=='roime81/trimorious-research-4B-v2'
    approved=get(run_dir/'package_approval.json'); assert approved['target']==TARGET and approved['evaluation_sha256']==sha(report)
    assert approved['rubric_review_sha256']==sha((eval_dir/'rubric_review_private.csv').read_bytes())
    assert approved['manifest_sha256']==sha((package/'artifact_manifest.json').read_bytes())
    manifest=get(package/'artifact_manifest.json'); assert all(sha((package/k).read_bytes())==v for k,v in manifest.items())
    assert set(p.name for p in package.iterdir())==set(manifest)|{'artifact_manifest.json'}
    import subprocess
    subprocess.check_call([sys.executable,'-m','pip','install','-q','huggingface_hub>=0.34,<1'])
    from huggingface_hub import HfApi,hf_hub_download
    from huggingface_hub.utils import RepositoryNotFoundError
    from google.colab import userdata
    token=userdata.get('HF_TOKEN'); assert token
    api=HfApi(token=token); assert api.whoami()['name']=='roime81', 'Wrong Hugging Face account.'
    receipt=run_dir/'publish_receipt.json'
    if not receipt.exists():
        api.create_repo(repo_id=TARGET,repo_type='model',private=True,exist_ok=False)
        put(receipt,{'target':TARGET,'manifest_sha256':approved['manifest_sha256'],'verified':False})
    rec=get(receipt); assert rec['target']==TARGET and rec['manifest_sha256']==approved['manifest_sha256']
    info=api.model_info(TARGET); assert info.private or rec['verified'], 'Unexpected public repository; stop.'
    names=set(manifest)|{'artifact_manifest.json'}
    remote=set(api.list_repo_files(TARGET,repo_type='model')); assert remote<=names|{'.gitattributes'}, 'Unexpected remote files; refusing to overwrite.'
    if not rec['verified']:
        api.upload_folder(repo_id=TARGET,repo_type='model',folder_path=str(package),allow_patterns=sorted(names),commit_message='Release Trimorious 2.0 experimental QLoRA adapter')
    remote_info=api.model_info(TARGET); commit=remote_info.sha; assert names<=set(api.list_repo_files(TARGET,revision=commit))
    for name in sorted(names):
        local=hf_hub_download(TARGET,name,revision=commit,token=token,force_download=True)
        assert sha(Path(local).read_bytes())==sha((package/name).read_bytes()),f'Remote hash mismatch: {name}'
    rec.update(verified=True,revision=commit,private=bool(remote_info.private)); put(receipt,rec)
    if MAKE_PUBLIC:
        assert PUBLIC_APPROVAL=='MAKE TRIMORIOUS 2.0 PUBLIC', 'Explicit public approval required.'
        api.update_repo_visibility(repo_id=TARGET,private=False,repo_type='model'); assert not api.model_info(TARGET).private
        rec['private']=False; put(receipt,rec)
    state('PUBLISH',complete=not rec['private'],private_uploaded_verified=True,url='https://huggingface.co/'+TARGET,revision=commit)
    print('Verified repository:', 'https://huggingface.co/'+TARGET,'| private:',rec['private'],'| revision:',commit)

## Execute stage
Read the actual outputs. A stopped or failed gate is not a completed stage.

In [ ]:
# @title Execute this notebook stage
def file_sha(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
def read_json(p): return json.loads(Path(p).read_text())
def save_json(p,x):
    p=Path(p); temp=p.with_suffix(p.suffix+'.tmp'); temp.write_text(json.dumps(x,indent=2)); temp.replace(p)
def record_controller(stage,status,detail):
    save_json(WORK/'auto_controller_state.json',{'stage':stage,'status':status,'detail':str(detail)})
def pause(stage,detail):
    record_controller(stage,'waiting',detail); print('\nPAUSED — '+stage+'\n'+str(detail)+'\nSave changes, then Runtime → Run all. Existing artifacts are preserved.',flush=True)
def cpu(stage, parts):
    ns={**CONFIG,'USE_DRIVE':False,'STAGE':stage,'__name__':'__main__'}
    for part in ['common']+parts:
        p=SCRIPT_DIR/(part+'.py'); exec(compile(p.read_text(),str(p),'exec'),ns)
    return ns

def gpu(stage):
    print('\n=== '+stage+' — isolated GPU worker ===',flush=True)
    record_controller(stage,'running','GPU subprocess')
    config_path=SCRIPT_DIR/'run_config.json'; save_json(config_path,CONFIG)
    env=os.environ.copy(); env['PYTHONUNBUFFERED']='1'; env.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
    p=subprocess.Popen([sys.executable,'-u',str(SCRIPT_DIR/'worker.py'),str(config_path),stage],env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in p.stdout:
            print(line,end='',flush=True)
        status=p.wait()
    except BaseException:
        p.terminate()
        try: p.wait(timeout=20)
        except subprocess.TimeoutExpired: p.kill(); p.wait()
        raise
    if status: raise RuntimeError(f'{stage} worker exited with code {status}. Read the actual error above; checkpoints preserved.')
    # Child exit releases CUDA allocations. Dependencies already installed for subsequent child processes in this session.
    CONFIG['INSTALL_GPU_DEPS']=False

def validate_active(ns):
    approval,run,done,final=ns['active_run']()
    cfg=done['config']
    for k,v in {'base':BASE,'revision':REV,'max_length':MAX_LENGTH,'grad_accum':GRAD_ACCUM,'epochs':EPOCHS,'seed':SEED}.items():
        assert cfg[k]==v,f'Trained adapter uses different {k}; do not silently change a finished run.'
    assert done['steps']>0 and (final/'adapter_model.safetensors').is_file()
    return approval,run,done,final

def get_eval_artifacts(run):
    active=read_json(WORK/'active_evaluation.json'); assert active['run']==run.name
    folder=run/'evaluation'/active['evaluation']; return folder,read_json(folder/'evaluation_report.json')

def eval_receipt(run,done):
    folder,report=get_eval_artifacts(run)
    assert report['evaluation_config']['adapter_hashes']==done['adapter_hashes']
    assert report['evaluation_config']['fresh_hash']==FRESH_TEST_SHA256 and report['fresh']
    db=folder/'responses.sqlite3'
    with sqlite3.connect(db.resolve().as_uri()+'?mode=ro',uri=True) as conn:
        assert conn.execute('PRAGMA integrity_check').fetchone()[0]=='ok'
        counts=dict(conn.execute('SELECT model,COUNT(*) FROM responses GROUP BY model'))
        expected=1000+report['fresh']['n']
        assert counts=={'qwen3_base':expected,'trimorious_v2':expected}, f'Incomplete paired evaluation: {counts}'
    return {'evaluation':folder.name,'report_sha256':file_sha(folder/'evaluation_report.json'),'db_sha256':file_sha(db),'mapping_sha256':file_sha(folder/'rubric_mapping_private.json'),'fresh_sha256':FRESH_TEST_SHA256}



def audit_stage(stage, paths):
    from datetime import datetime, timezone
    target=WORK/'stage_audits'; target.mkdir(exist_ok=True)
    files=[Path(p) for p in paths if Path(p).is_file()]
    for p in files: assert p.resolve().is_relative_to(WORK.resolve())
    manifest={str(p.relative_to(WORK)):file_sha(p) for p in files}
    stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    dest=target/f'{stage}_{stamp}_PRIVATE.zip'; temp=dest.with_suffix('.tmp')
    import zipfile
    with zipfile.ZipFile(temp,'w',zipfile.ZIP_DEFLATED) as z:
        for p in files: z.write(p,str(p.relative_to(WORK)))
        z.writestr('audit_manifest.json',json.dumps(manifest,indent=2))
    with zipfile.ZipFile(temp) as z:
        assert z.testzip() is None
        assert all(hashlib.sha256(z.read(k)).hexdigest()==v for k,v in manifest.items())
    temp.replace(dest); print('Private stage audit:',dest)

ns=cpu('RECOVER',['approve','eval_setup']); approval,run,done,final=ns['active_run']()
folder,report=get_eval_artifacts(run); FRESH_TEST_SHA256=report['evaluation_config']['fresh_hash']
assert read_json(run/'auto_eval_receipt.json')==eval_receipt(run,done), 'Evaluation artifacts changed.'
assert (run/'package_approval.json').exists(), 'Run notebook 06 first.'
if not (DATA_LICENSE_REVIEWED and HUMAN_EVAL_APPROVED and PUBLISH_APPROVAL=='PUBLISH TRIMORIOUS 2.0'):
    pause(STAGE,'Inspect package, reconfirm rights and review, then explicitly approve private upload.')
elif MAKE_PUBLIC and (not (run/'publish_receipt.json').exists() or not read_json(run/'publish_receipt.json').get('verified') or PUBLIC_APPROVAL!='MAKE TRIMORIOUS 2.0 PUBLIC'):
    pause(STAGE,'Verified private upload and explicit public approval required first.')
else:
    cpu('PUBLISH',['approve','eval_setup','package','publish'])
    receipt=read_json(run/'publish_receipt.json'); assert receipt['verified']
    audit_stage(STAGE,[WORK/'project_state.json',run/'publish_receipt.json',run/'package_approval.json'])
    print('Verified private upload; inspect before approving public visibility.' if receipt['private'] else 'PUBLIC RELEASE VERIFIED: https://huggingface.co/'+TARGET)